# 📊 UAS Data Science — Week 2 Progress
## Data Preprocessing & Feature Engineering

**Dataset:** Sales & Marketing Customer Dataset  
**Tujuan Week 2:** Membersihkan data, menangani missing values, membuat fitur baru, dan menyiapkan data siap modeling

---

## 1. Recap Week 1

Dari hasil EDA di Week 1, ditemukan beberapa masalah yang perlu ditangani:

| Masalah | Kolom | Rencana Penanganan |
|---------|-------|--------------------|
| Missing values | `coupon_code` (40.9%), `age` (8%), `gender` (4.9%), `total_spent` (7%), `satisfaction_score` (4.7%) | Imputasi median/modus, fill 'No Coupon' |
| Anomali usia | `age` < 0 (3 baris), `age` > 100 | Drop baris anomali |
| Class imbalance | `churn`: 84.7% vs 15.3% | SMOTE pada tahap modeling |
| Kolom tanggal | `signup_date`, `last_purchase_date` | Ekstrak fitur numerik baru |
| Kolom kategorik | 7 kolom string | Label/One-Hot Encoding |

---

## 2. Import Library & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='Set2')

import warnings
warnings.filterwarnings('ignore')

print('✅ Library berhasil diimport')

In [ ]:
df = pd.read_csv('Sales_-_Marketing_customer_dataset.csv')
print(f'Dataset dimuat: {df.shape[0]:,} baris, {df.shape[1]} kolom')
df.head(3)

## 3. Data Cleaning

### 3.1 Menghapus Baris dengan Anomali Usia

In [ ]:
print(f'Jumlah baris sebelum cleaning: {len(df):,}')

# Tampilkan baris anomali
anomali_age = df[(df['age'] < 0) | (df['age'] > 100)]
print(f'Baris dengan age tidak wajar  : {len(anomali_age)}')
print(anomali_age[['customer_id', 'age', 'churn']])

# Hapus baris anomali
df = df[~((df['age'] < 0) | (df['age'] > 100))].copy()
df.reset_index(drop=True, inplace=True)

print(f'\nJumlah baris setelah cleaning : {len(df):,}')

### 3.2 Menangani Missing Values

In [ ]:
print('Missing values SEBELUM imputasi:')
missing_before = df.isnull().sum()
print(missing_before[missing_before > 0])
print()

In [ ]:
# Imputasi coupon_code: NaN berarti tidak menggunakan kupon
df['coupon_code'] = df['coupon_code'].fillna('No Coupon')

# Imputasi gender dengan modus
df['gender'] = df['gender'].fillna(df['gender'].mode()[0])

# Imputasi age dengan median (robust terhadap outlier)
median_age = df['age'].median()
df['age'] = df['age'].fillna(median_age)
print(f'Median age untuk imputasi: {median_age}')

# Imputasi total_spent dengan median
median_spent = df['total_spent'].median()
df['total_spent'] = df['total_spent'].fillna(median_spent)
print(f'Median total_spent untuk imputasi: {median_spent:.2f}')

# Imputasi satisfaction_score dengan median
median_sat = df['satisfaction_score'].median()
df['satisfaction_score'] = df['satisfaction_score'].fillna(median_sat)
print(f'Median satisfaction_score untuk imputasi: {median_sat}')

print()
print('Missing values SETELAH imputasi:', df.isnull().sum().sum())
print('✅ Semua missing values berhasil ditangani')

## 4. Feature Engineering

### 4.1 Ekstraksi Fitur dari Kolom Tanggal

In [ ]:
# Parse tanggal
df['signup_date'] = pd.to_datetime(df['signup_date'])
df['last_purchase_date'] = pd.to_datetime(df['last_purchase_date'])

# Reference date: awal 2025
reference_date = pd.Timestamp('2025-01-01')

# Fitur baru dari tanggal
df['customer_tenure_days'] = (reference_date - df['signup_date']).dt.days
df['recency_days']         = (reference_date - df['last_purchase_date']).dt.days
df['signup_year']          = df['signup_date'].dt.year
df['signup_month']         = df['signup_date'].dt.month

print('Fitur tanggal berhasil dibuat:')
print(df[['customer_id','signup_date','last_purchase_date',
          'customer_tenure_days','recency_days','signup_year']].head(5))

### 4.2 Membuat Fitur Turunan (Derived Features)

In [ ]:
# Pengeluaran per kunjungan
df['spend_per_visit'] = df['total_spent'] / (df['total_visits'] + 1)

# Skor engagement gabungan dari email
df['engagement_score'] = (df['email_open_rate'] * 0.5) + (df['email_click_rate'] * 0.5)

# Rasio refund terhadap total tiket support
df['refund_rate'] = df['refund_requested'] / (df['support_tickets'] + 1)

new_features = ['spend_per_visit', 'engagement_score', 'refund_rate',
                'customer_tenure_days', 'recency_days']

print('Fitur baru yang dibuat:')
print(df[new_features].describe().T[['mean','std','min','max']])
print()
print(f'Total kolom sekarang: {df.shape[1]}')

In [ ]:
# Visualisasi distribusi fitur baru
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, feat in zip(axes, ['spend_per_visit', 'engagement_score', 'recency_days']):
    for label, color in zip([0, 1], ['#2ecc71', '#e74c3c']):
        ax.hist(df[df['churn'] == label][feat], bins=40, alpha=0.6,
                color=color, density=True,
                label='Tidak Churn' if label == 0 else 'Churn')
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold')
    ax.legend()

plt.suptitle('Distribusi Fitur Baru: Churn vs Tidak Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Encoding Variabel Kategorik

### 5.1 Identifikasi Kolom yang Perlu Di-encode

In [ ]:
# Kolom yang akan di-drop (tidak dipakai untuk modeling)
drop_cols = ['customer_id', 'signup_date', 'last_purchase_date', 'city']
df_model = df.drop(columns=drop_cols).copy()

# Identifikasi kolom kategorik yang tersisa
cat_cols = ['gender', 'country', 'acquisition_channel', 'device_type',
            'subscription_type', 'coupon_code', 'payment_method']

print('Kolom yang akan di-encode (Label Encoding):')
for col in cat_cols:
    print(f'  {col}: {df_model[col].unique().tolist()}')

In [ ]:
# Label Encoding
le = LabelEncoder()
label_encoders = {}

for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le

print('✅ Label Encoding selesai')
print(f'Shape setelah encoding: {df_model.shape}')
df_model.head(3)

## 6. Feature Scaling

### 6.1 Persiapan Fitur & Target

In [ ]:
# Pisahkan fitur dan target
X = df_model.drop(columns=['churn'])
y = df_model['churn']

print(f'Shape X (fitur): {X.shape}')
print(f'Shape y (target): {y.shape}')
print(f'Distribusi target:')
print(y.value_counts())

In [ ]:
# Kolom yang perlu di-scale (numerik kontinu)
scale_cols = ['age', 'total_visits', 'avg_session_time', 'pages_per_session',
              'email_open_rate', 'email_click_rate', 'total_spent', 'avg_order_value',
              'delivery_delay_days', 'satisfaction_score', 'nps_score',
              'marketing_spend_per_user', 'lifetime_value', 'last_3_month_purchase_freq',
              'customer_tenure_days', 'recency_days', 'spend_per_visit',
              'engagement_score', 'refund_rate']

scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[scale_cols] = scaler.fit_transform(X[scale_cols])

print('✅ StandardScaler diterapkan pada kolom numerik kontinu')
print(f'Contoh nilai setelah scaling (5 kolom pertama):')
print(X_scaled[scale_cols[:5]].describe().T[['mean','std']].round(4))

## 7. Visualisasi Data Bersih

In [ ]:
# Distribusi age sebelum vs sesudah cleaning
df_raw = pd.read_csv('Sales_-_Marketing_customer_dataset.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_raw['age'].dropna(), bins=40, color='#e74c3c', alpha=0.7, edgecolor='white')
axes[0].set_title('Distribusi Age — SEBELUM Cleaning', fontweight='bold')
axes[0].set_xlabel('Age')

axes[1].hist(df['age'], bins=40, color='#2ecc71', alpha=0.7, edgecolor='white')
axes[1].set_title('Distribusi Age — SETELAH Cleaning', fontweight='bold')
axes[1].set_xlabel('Age')

plt.suptitle('Perbandingan Distribusi Age Sebelum & Sesudah Cleaning', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Korelasi fitur baru dengan churn
new_feat_corr = df_model[['spend_per_visit','engagement_score','recency_days',
                           'customer_tenure_days','refund_rate','churn']].corr()['churn'].drop('churn')

colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in new_feat_corr.values]

plt.figure(figsize=(9, 4))
bars = plt.barh(new_feat_corr.index, new_feat_corr.values, color=colors)
plt.axvline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, new_feat_corr.values):
    plt.text(val + (0.001 if val >= 0 else -0.001), bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', ha='left' if val >= 0 else 'right', fontsize=10)
plt.title('Korelasi Fitur Baru dengan Target Churn', fontsize=13, fontweight='bold')
plt.xlabel('Korelasi (Pearson)')
plt.tight_layout()
plt.show()

## 8. Simpan Data yang Sudah Diproses

In [ ]:
# Simpan dataset bersih (sebelum scaling) untuk referensi
df_model.to_csv('data_clean.csv', index=False)

# Simpan dataset siap modeling (sudah di-scale)
X_scaled['churn'] = y.values
X_scaled.to_csv('data_modeling.csv', index=False)

print('✅ File berhasil disimpan:')
print('   - data_clean.csv    : data bersih + fitur baru (belum di-scale)')
print('   - data_modeling.csv : data siap modeling (sudah di-scale)')
print(f'\nShape final: {X_scaled.shape}')
print(f'Total fitur: {X_scaled.shape[1] - 1} (tidak termasuk target)')

## 9. Ringkasan Week 2

### ✅ Yang Telah Dilakukan

| Tahap | Detail |
|-------|--------|
| **Cleaning** | Hapus 3 baris age negatif → data menjadi 14.997 baris |
| **Imputasi** | Median untuk age/total_spent/satisfaction_score; Modus untuk gender; 'No Coupon' untuk coupon_code |
| **Feature Engineering** | 5 fitur baru: `customer_tenure_days`, `recency_days`, `signup_year`, `spend_per_visit`, `engagement_score`, `refund_rate` |
| **Encoding** | Label Encoding pada 7 kolom kategorik |
| **Scaling** | StandardScaler pada 19 kolom numerik kontinu |

### 🗓️ Rencana Week 3
- Train-test split (80:20)
- Penanganan class imbalance dengan **SMOTE**
- Baseline modeling: **Logistic Regression**, **Decision Tree**, **Random Forest**
- Evaluasi: Accuracy, Precision, Recall, F1-Score, ROC-AUC
- Perbandingan performa antar model